# Seismic Hotspot Detection & Evolution
Phase 6: DBSCAN & K-Means Clustering | Phase 7: Temporal Hotspot Evolution

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid')

BASE_DIR = os.path.abspath('..')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
df = pd.read_csv(os.path.join(OUTPUT_DIR, 'cleaned_earthquake_data.csv'), parse_dates=['Datetime'])
print(f'Loaded {len(df)} records.')

## DBSCAN Hotspot Detection

In [ ]:
coords = df[['Latitude','Longitude']].values
scaler = StandardScaler()
coords_scaled = scaler.fit_transform(coords)

dbscan = DBSCAN(eps=0.2, min_samples=10)
df['DBSCAN_Cluster'] = dbscan.fit_predict(coords_scaled)

n_clusters = len(set(df['DBSCAN_Cluster'])) - (1 if -1 in df['DBSCAN_Cluster'].values else 0)
n_noise = (df['DBSCAN_Cluster'] == -1).sum()
print(f'DBSCAN: {n_clusters} clusters | {n_noise} noise points ({n_noise/len(df)*100:.1f}%)')

In [ ]:
plt.figure(figsize=(12, 8))
palette = sns.color_palette('tab10', n_colors=max(n_clusters, 1))
for cid in sorted(df['DBSCAN_Cluster'].unique()):
    sub = df[df['DBSCAN_Cluster'] == cid]
    label = 'Noise' if cid == -1 else f'Cluster {cid}'
    color = 'lightgrey' if cid == -1 else palette[min(cid, len(palette)-1)]
    plt.scatter(sub['Longitude'], sub['Latitude'], label=label, s=10, alpha=0.6, c=[color]*len(sub))
plt.legend(loc='lower right', fontsize=9, markerscale=2)
plt.title('DBSCAN Seismic Hotspots (grey = noise)', fontsize=14)
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.savefig(os.path.join(OUTPUT_DIR, 'dbscan_hotspots.png'), dpi=150)
plt.show()

## K-Means: Elbow Method

In [ ]:
inertias = []
k_range = range(2, 10)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    km.fit(coords_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_range), inertias, marker='o', color='green', linewidth=2)
plt.title('K-Means Elbow Method')
plt.xlabel('Number of Clusters (K)'); plt.ylabel('Inertia')
plt.savefig(os.path.join(OUTPUT_DIR, 'kmeans_elbow.png'), dpi=150)
plt.show()

In [ ]:
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init='auto')
df['KMeans_Cluster'] = kmeans.fit_predict(coords_scaled)

plt.figure(figsize=(12, 8))
sns.scatterplot(data=df, x='Longitude', y='Latitude',
                hue='KMeans_Cluster', palette='Set1', alpha=0.6, s=12)
plt.title(f'K-Means Clustering (K={optimal_k})', fontsize=14)
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.savefig(os.path.join(OUTPUT_DIR, 'kmeans_hotspots.png'), dpi=150)
plt.show()

## Why DBSCAN is Better for Seismic Data

In [ ]:
explanation = """
DBSCAN vs K-Means for Seismic Hotspot Detection
================================================
1. No need to specify cluster count — DBSCAN discovers it automatically.
2. Finds arbitrary shapes — fault lines are linear/curved, not spherical like K-Means assumes.
3. Handles noise naturally — isolated events are labelled as noise (cluster -1).
4. Robust to outliers — extreme aftershock sequences don't distort cluster centers.
"""
print(explanation)
with open(os.path.join(OUTPUT_DIR, 'clustering_comparison.txt'), 'w', encoding='utf-8') as f:
    f.write(explanation)

## Hotspot Evolution Over Time (1990–2026)

In [ ]:
periods = ['1990-1999','2000-2009','2010-2019','2020-2026']
fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharex=True, sharey=True)
axes = axes.flatten()

for idx, period in enumerate(periods):
    pdf = df[df['Period'] == period].copy()
    ax = axes[idx]
    if len(pdf) < 10:
        ax.set_title(f'{period} (insufficient data)')
        continue
    sc = StandardScaler()
    pc = sc.fit_transform(pdf[['Latitude','Longitude']].values)
    c = DBSCAN(eps=0.25, min_samples=5).fit_predict(pc)
    pdf = pdf.copy()
    pdf['PCluster'] = c
    n_c = len(set(c)) - (1 if -1 in c else 0)
    palette = sns.color_palette('tab10', n_colors=max(n_c, 1))
    for cid in sorted(pdf['PCluster'].unique()):
        sub = pdf[pdf['PCluster'] == cid]
        col = 'lightgrey' if cid == -1 else palette[min(cid, len(palette)-1)]
        ax.scatter(sub['Longitude'], sub['Latitude'], s=8, alpha=0.6, c=[col]*len(sub))
    ax.set_title(f'{period}  |  Clusters: {n_c}  |  Events: {len(pdf)}')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

plt.suptitle('Seismic Hotspot Evolution by Period', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'hotspot_evolution.png'), dpi=150)
plt.show()
# Save updated dataframe with cluster columns
df.to_csv(os.path.join(OUTPUT_DIR, 'clustered_earthquake_data.csv'), index=False)
print('Saved clustered dataset.')